# B3 — обучение на Google Colab (T4 GPU)

Переобучение **Stage-2 EfficientNet (48 классов (43 GTSRB + 5 RU))** и опционально **Stage-1 YOLO**.

## Что нужно один раз сделать ДО запуска
Залить на свой Google Drive один архив **`mitgo_data.zip`**, внутри — три папки
(имена не важны, найдём автоматически по содержимому):

* Kaggle GTSRB — папка, где лежат `Train.csv` / `Test.csv` / `Train/` / `Test/`
* `rtsd_crops/` — `{train,val}/{43..47}/...jpg` (из `tools/rtsd_to_crops.py`)
* `gtsdb_yolo/` — `images/ labels/ gtsdb.yaml` (только если будете дообучать YOLO)

Runtime → Change runtime type → **GPU (T4)**, затем выполняйте ячейки сверху вниз.

In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, 'CUDA', torch.cuda.is_available())

In [ ]:
# Clone the repo (Stage-2 / tooling lives here)
%cd /content
![ -d MITGOTSD ] || git clone https://github.com/damir095/MITGOTSD.git
%cd /content/MITGOTSD
!git pull --ff-only || true

In [ ]:
# Deps. Colab already has a CUDA torch — do NOT reinstall torch.
!pip -q install ultralytics timm scikit-learn opencv-python-headless pillow pandas tqdm

In [ ]:
# Mount Drive and unzip the data once into /content/datasets.
# Guard on a MARKER (rtsd_crops/train/47 — the last RU class), not "is the
# dir non-empty": a partial earlier extract once left only gtsdb_yolo here
# and the old non-empty check silently skipped the full re-extract.
from google.colab import drive
drive.mount('/content/drive')

DATA_ZIP  = '/content/drive/MyDrive/mitgo_data.zip'   # <-- edit if needed
DATA_ROOT = '/content/datasets'

import zipfile, pathlib, glob, shutil
marker = glob.glob(f'{DATA_ROOT}/**/rtsd_crops/train/47', recursive=True)
if not marker:
    shutil.rmtree(DATA_ROOT, ignore_errors=True)
    pathlib.Path(DATA_ROOT).mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATA_ZIP) as z:
        z.extractall(DATA_ROOT)
    print('extracted into', DATA_ROOT)
else:
    print('already extracted (marker rtsd_crops/train/47 present)')

cls = sorted(int(pathlib.Path(p).name) for p in
             glob.glob(f'{DATA_ROOT}/**/rtsd_crops/train/*', recursive=True)
             if pathlib.Path(p).is_dir())
print('rtsd_crops train classes:', cls)
assert cls == [43, 44, 45, 46, 47], (
    f'STALE ZIP: expected 5 RU classes 43-47, got {cls}. '
    'Rebuild mitgo_data.zip locally and re-upload to Drive.')

In [ ]:
# Auto-locate the datasets (robust to however the zip was structured)
import glob, os, pathlib

train_csv = glob.glob(f'{DATA_ROOT}/**/Train.csv', recursive=True)
rtsd_dir  = glob.glob(f'{DATA_ROOT}/**/rtsd_crops', recursive=True)
yaml_path = glob.glob(f'{DATA_ROOT}/**/gtsdb.yaml', recursive=True)
assert train_csv, 'Train.csv not found under DATA_ROOT — check the zip'

os.environ['GTSRB_KAGGLE_ROOT'] = str(pathlib.Path(train_csv[0]).parent)
if rtsd_dir:
    os.environ['RTSD_CROPS_ROOT'] = rtsd_dir[0]
print('GTSRB_KAGGLE_ROOT =', os.environ['GTSRB_KAGGLE_ROOT'])
print('RTSD_CROPS_ROOT  =', os.environ.get('RTSD_CROPS_ROOT', '(none — 43-class only)'))
print('gtsdb.yaml       =', yaml_path[0] if yaml_path else '(none — skip YOLO cell)')

## Stage-2 — EfficientNet (48 классов (43 GTSRB + 5 RU))
Сначала smoke (1+1 эпоха) — проверить, что цепочка цела, потом полный прогон.

In [ ]:
# Smoke: must reach the GTSRB-test report + 'OUT-OF-DOMAIN HOLD-OUT' skip line
!cd /content/MITGOTSD && GTSRB_EPOCHS_HEAD=1 GTSRB_EPOCHS_FULL=1 GTSRB_BATCH_SIZE=64 python train.py

In [ ]:
# Full run. T4 16 GB -> big batch, fast. ~ a couple of hours.
!cd /content/MITGOTSD && GTSRB_BATCH_SIZE=256 GTSRB_NUM_WORKERS=2 python train.py

In [ ]:
# Save the 48-class classifier back to Drive (and offer a direct download)
import shutil
src = '/content/MITGOTSD/experiments/checkpoints/best.pt'
shutil.copy(src, '/content/drive/MyDrive/efficientnet_ru48.pt')
print('copied to Drive: efficientnet_ru48.pt')
from google.colab import files; files.download(src)

## Stage-1 — YOLO (опционально, только если залили `gtsdb_yolo/`)
`gtsdb.yaml` содержит абсолютный путь с Windows-машины — чиним под Colab.

In [ ]:
# Rewrite gtsdb.yaml 'path:' to the Colab location
assert yaml_path, 'gtsdb.yaml not found — upload gtsdb_yolo/ to retrain YOLO'
yp = yaml_path[0]
lines = pathlib.Path(yp).read_text().splitlines()
lines = [f'path: {pathlib.Path(yp).parent.as_posix()}' if l.startswith('path:') else l
         for l in lines]
pathlib.Path(yp).write_text('\n'.join(lines) + '\n')
print(pathlib.Path(yp).read_text())

In [ ]:
# T4 8GB+ -> bigger imgsz/batch than the MX330 (which was 416/4)
!cd /content/MITGOTSD && yolo detect train model=yolov8n.pt data={yp} \
    imgsz=640 batch=32 epochs=100 device=0 project=experiments/yolo name=gtsdb

In [ ]:
import shutil, glob
bp = glob.glob('/content/MITGOTSD/runs/detect/**/best.pt', recursive=True)[0]
shutil.copy(bp, '/content/drive/MyDrive/yolo_gtsdb.pt')
print('copied to Drive: yolo_gtsdb.pt  (from', bp, ')')
from google.colab import files; files.download(bp)

## Назад на локальную машину
Скопировать `efficientnet_ru48.pt` → `project/experiments/checkpoints/best.pt`
и (если переобучали) `yolo_gtsdb.pt` →
`project/runs/detect/experiments/yolo/gtsdb/weights/best.pt`.
Затем локально: `python -m src.evaluate_b3` и `python -m src.yolo_pipeline`.